In [1]:
import warnings

import numpy as np
import pandas as pd
from pandas.errors import PerformanceWarning
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

RNG_SEED = 42
warnings.filterwarnings("ignore", category=PerformanceWarning)
warnings.filterwarnings("ignore", message=".*Parameters:.*use_label_encoder.*")
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost.training")


In [2]:
# -----------------------------
# 1) Load cleaned data
# -----------------------------
train = pd.read_csv("train_clean.csv")
test = pd.read_csv("test_clean.csv")

print("Raw train shape:", train.shape)
print("Raw test shape:", test.shape)
print(f"Target rate: {train['TARGET'].mean():.5f}")
train.head()


Raw train shape: (76020, 308)
Raw test shape: (75818, 307)
Target rate: 0.03957


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [3]:
# -----------------------------
# 2) Feature Engineering: activity/sparsity + var38
# -----------------------------
target_col = "TARGET"
id_col = "ID"

base_feature_cols = [c for c in train.columns if c not in [id_col, target_col]]

# Activity / sparsity features
train["zero_count"] = (train[base_feature_cols] == 0).sum(axis=1)
test["zero_count"] = (test[base_feature_cols] == 0).sum(axis=1)

train["nonzero_count"] = (train[base_feature_cols] != 0).sum(axis=1)
test["nonzero_count"] = (test[base_feature_cols] != 0).sum(axis=1)

# var38 special features
peak = 117310.979016494

if "var38" in train.columns:
    train["var38_is_peak"] = (train["var38"] == peak).astype(int)
    test["var38_is_peak"] = (test["var38"] == peak).astype(int)

    train["var38_log"] = np.log1p(train["var38"])
    test["var38_log"] = np.log1p(test["var38"])

    train.loc[train["var38"] == peak, "var38_log"] = 0
    test.loc[test["var38"] == peak, "var38_log"] = 0

# Final feature columns after FE
feature_cols = [c for c in train.columns if c not in [id_col, target_col]]

X = train[feature_cols]
y = train[target_col]
X_test = test[feature_cols]

test_ids = test[id_col]

print("Train shape after FE:", X.shape)
print("Test shape after FE:", X_test.shape)
print("New features added:", ["zero_count", "nonzero_count", "var38_is_peak", "var38_log"])


Train shape after FE: (76020, 310)
Test shape after FE: (75818, 310)
New features added: ['zero_count', 'nonzero_count', 'var38_is_peak', 'var38_log']


In [4]:
# -----------------------------
# 3) Sanity checks
# -----------------------------
print(X.shape, X_test.shape)
print(set(X.columns) - set(X_test.columns))
print(set(X_test.columns) - set(X.columns))
print(X[["zero_count", "nonzero_count", "var38_log", "var38_is_peak"]].head())


(76020, 310) (75818, 310)
set()
set()
   zero_count  nonzero_count  var38_log  var38_is_peak
0         292             14  10.576589              0
1         266             40  10.805254              0
2         277             29  11.117432              0
3         248             58  11.066779              0
4         256             50   0.000000              1


In [5]:
# -----------------------------
# 4) 5-fold Stratified CV with unchanged XGBoost cleaned_v1 hyperparameters
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)

oof_pred = np.zeros(len(train))
test_pred = np.zeros(len(test))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        use_label_encoder=False,
        random_state=RNG_SEED + fold,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    valid_pred = model.predict_proba(X_valid)[:, 1]
    oof_pred[valid_idx] = valid_pred

    fold_auc = roc_auc_score(y_valid, valid_pred)
    fold_scores.append(fold_auc)

    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits

    print(f"Fold {fold} AUC: {fold_auc:.5f}")

oof_auc = roc_auc_score(y, oof_pred)

print("=" * 40)
print("Fold AUCs:", fold_scores)
print(f"OOF AUC: {oof_auc:.5f}")


Fold 1 AUC: 0.83063


Fold 2 AUC: 0.84173


Fold 3 AUC: 0.83923


Fold 4 AUC: 0.83471


Fold 5 AUC: 0.84227
Fold AUCs: [0.8306296440580497, 0.8417251919721553, 0.8392331000941481, 0.8347143089214102, 0.8422690242678266]
OOF AUC: 0.83764


In [6]:
# -----------------------------
# 5) Generate submission
# -----------------------------
submission = pd.DataFrame({
    "ID": test_ids,
    "TARGET": test_pred,
})

submission.to_csv("submission_xgboost_fe.csv", index=False)
print("Saved: submission_xgboost_fe.csv")
submission.head()


Saved: submission_xgboost_fe.csv


,ID,TARGET
0,2,0.052818
1,5,0.058325
2,6,0.001025
3,7,0.008204
4,9,0.001233


In [7]:
# -----------------------------
# 6) Record result row
# -----------------------------
xgboost_fe_result = pd.DataFrame([
    {
        "Model": "XGBoost FE",
        "Private Score": "?",
        "Public Score": "?",
        "OOF AUC": round(oof_auc, 5),
        "Hyperparameters": "same as XGBoost cleaned_v1",
        "Notes": "Added zero_count, nonzero_count, var38_log, var38_is_peak",
    }
])

xgboost_fe_result.to_csv("xgboost_fe_results.csv", index=False)
xgboost_fe_result


,Model,Private Score,Public Score,OOF AUC,Hyperparameters,Notes
0,XGBoost FE,?,?,0.83764,same as XGBoost cleaned_v1,"Added zero_count, nonzero_count, var38_log, va..."
